In [5]:
import glob
import json
import os
import re

import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [4]:

DC8_RE = re.compile(r"asiaaq_(\d{8})_dc8_.*\.nc4$")
TROP_RE = re.compile(r"asiaaq_(\d{8})_tropomi_l2_([a-z0-9]+)_.*\.nc4$")

# light physical sanity clips per species (obs column, molecules/cm2).
# matches the clipping used in the MM plot_sat.py driver script.
CLIP = {
    "no2": (-2e15, 5e16),
    "hcho": (-2e16, 5e16),
    "co": (0.0, 1e19),
}

def _dict_json(ds):
    """Return the parsed MELODIES-MONET dict_json global attr, or {}."""
    raw = ds.attrs.get("dict_json")
    if not raw:
        return {}
    try:
        return json.loads(raw)
    except (ValueError, TypeError):
        return {}


def _obs_var(ds):
    """Name of the observation data variable in a TROPOMI pair file."""
    meta = _dict_json(ds)
    obs_vars = meta.get("obs_vars") or []
    model_vars = set(meta.get("model_vars") or [])
    for v in obs_vars:
        if v in ds.data_vars:
            return v
    # fallback: a (time, n_face) var that is not a declared model var
    for v in ds.data_vars:
        if v not in model_vars and set(ds[v].dims) >= {"time", "n_face"}:
            return v
    raise KeyError(f"could not find an obs column variable in {list(ds.data_vars)}")


def _wrap180(lon):
    """Normalize longitudes to [-180, 180) so both files share a convention."""
    return (np.asarray(lon, dtype=float) + 180.0) % 360.0 - 180.0


def _squeeze_track(ds):
    """Pull lon/lat/alt/time 1-D arrays out of a DC8 pair (time, x=1)."""
    d = ds.squeeze(drop=False)
    lon = _wrap180(np.asarray(d["longitude"].values, float).ravel())
    lat = np.asarray(d["latitude"].values, float).ravel()
    alt = (np.asarray(d["altitude"].values, float).ravel()
           if "altitude" in d else np.full_like(lat, np.nan))
    t = np.asarray(d["time"].values)  # datetime64 after CF decode
    return lon, lat, alt, t


# core
def find_pairs(pairdir):
    """{date: {'dc8': path, 'tropomi': {species: path}}} for dates having both."""
    dc8 = {}
    trop = {}
    for p in sorted(glob.glob(os.path.join(pairdir, "asiaaq_*.nc4"))):
        base = os.path.basename(p)
        m = DC8_RE.match(base)
        if m:
            dc8[m.group(1)] = p
            continue
        m = TROP_RE.match(base)
        if m:
            trop.setdefault(m.group(1), {})[m.group(2)] = p
    out = {}
    for date in sorted(set(dc8) & set(trop)):
        out[date] = {"dc8": dc8[date], "tropomi": trop[date]}
    return out


def coincident_overpasses(flight_t, overpass_t, window_hours):
    """For each overpass, the DC8 sample mask within +/-window and the min dt.

    Returns list of dicts (one per overpass) sorted by ascending min dt.
    """
    win = np.timedelta64(int(window_hours * 3600), "s")
    rows = []
    for k, ot in enumerate(overpass_t):
        dt = np.abs(flight_t - ot)  # timedelta64 array
        in_win = dt <= win
        finite = np.isfinite(flight_t.astype("float64"))
        n_in = int(np.count_nonzero(in_win & finite))
        min_dt = dt[finite].min() if finite.any() else np.timedelta64(10**9, "s")
        rows.append({
            "k": k,
            "overpass_time": ot,
            "mask_in_window": in_win,
            "n_in_window": n_in,
            "min_dt_min": float(min_dt / np.timedelta64(1, "m")),
        })
    rows.sort(key=lambda r: r["min_dt_min"])
    return rows


def render(date, species, obs, lon_f, lat_f, alt_f, t_f, ov, extent,
           vmin, vmax, units, outpath, field_label):
    """One overlay figure: TROPOMI column point-map + DC8 track."""
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(11, 9))
    ax = fig.add_subplot(1, 1, 1, projection=proj)

    ax.coastlines(linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.STATES, linewidth=0.25, edgecolor="gray")
    gl = ax.gridlines(draw_labels=True, lw=0.4, color="gray", alpha=0.4, ls=":")
    gl.top_labels = False
    gl.right_labels = False
    ax.set_extent(extent, crs=proj)

    # --- TROPOMI column background (regridded mesh faces where finite) -------
    lon_s, lat_s, val_s = obs
    finite = np.isfinite(val_s)
    sc = ax.scatter(lon_s[finite], lat_s[finite], c=val_s[finite],
                    s=9, marker="s", cmap="Spectral_r",
                    vmin=vmin, vmax=vmax, linewidths=0, rasterized=True,
                    transform=proj, zorder=1)
    cb = fig.colorbar(sc, ax=ax, shrink=0.75, aspect=28, pad=0.02, extend="both")
    cb.set_label(f"TROPOMI {species.upper()} {field_label}  [{units}]",
                 fontweight="bold")

    # --- DC8 flight track (plain path + markers) -----------------------------
    ok = np.isfinite(lon_f) & np.isfinite(lat_f)
    ax.plot(lon_f[ok], lat_f[ok], "-", color="black", lw=1.4,
            transform=proj, zorder=3, label="DC8 track")
    # airborne within the overpass window -> "under the overpass"
    win_ok = ok & ov["mask_in_window"]
    if win_ok.any():
        ax.scatter(lon_f[win_ok], lat_f[win_ok], s=22, facecolors="none",
                   edgecolors="black", linewidths=0.9, transform=proj,
                   zorder=4, label=f"DC8 within ±window ({win_ok.sum()} pts)")
    # start / end markers
    idx = np.where(ok)[0]
    if idx.size:
        ax.scatter(lon_f[idx[0]], lat_f[idx[0]], s=90, marker="o",
                   color="lime", edgecolors="black", zorder=5, label="takeoff")
        ax.scatter(lon_f[idx[-1]], lat_f[idx[-1]], s=90, marker="X",
                   color="red", edgecolors="black", zorder=5, label="landing")

    ax.legend(loc="upper right", fontsize=9, framealpha=0.9)

    ot = np.datetime_as_string(ov["overpass_time"], unit="m")
    ax.set_title(
        f"ASIA-AQ {date}  |  TROPOMI {species.upper()} overpass {ov['k']} @ {ot}Z\n"
        f"DC8 nearest approach {ov['min_dt_min']:.0f} min, "
        f"{ov['n_in_window']} samples in window",
        fontweight="bold")

    fig.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.close(fig)


def process_date(date, entry, args):
    print(f"\n=== {date} ===", flush=True)
    with xr.open_dataset(entry["dc8"]) as dc8:
        lon_f, lat_f, alt_f, t_f = _squeeze_track(dc8)
    fin = np.isfinite(lon_f) & np.isfinite(lat_f)
    if not fin.any():
        print("  DC8: no finite lon/lat, skipping date.", flush=True)
        return 0
    # flight bounding box (padded) -> map extent
    padx = max((np.nanmax(lon_f[fin]) - np.nanmin(lon_f[fin])) * 0.12, 0.3)
    pady = max((np.nanmax(lat_f[fin]) - np.nanmin(lat_f[fin])) * 0.12, 0.3)
    extent = [np.nanmin(lon_f[fin]) - padx, np.nanmax(lon_f[fin]) + padx,
              np.nanmin(lat_f[fin]) - pady, np.nanmax(lat_f[fin]) + pady]

    n_made = 0
    for species, path in sorted(entry["tropomi"].items()):
        with xr.open_dataset(path) as ds:
            ovar = _obs_var(ds)
            field = ovar if args.field == "obs" else (
                _dict_json(ds).get("model_vars", [ovar])[0])
            lon_s = _wrap180(ds["longitude"].values)
            lat_s = np.asarray(ds["latitude"].values, float)
            over_t = np.asarray(ds["time"].values)
            val_all = np.asarray(ds[field].values, float)  # (time, n_face)
            units = ds[ovar].attrs.get("units", "")

        # sanity clip
        lo, hi = CLIP.get(species, (-np.inf, np.inf))
        val_all = np.where((val_all > lo) & (val_all < hi), val_all, np.nan)

        # shared color scale across this species' overpasses (2-98 pct)
        finite_vals = val_all[np.isfinite(val_all)]
        if finite_vals.size == 0:
            print(f"  {species}: no finite obs pixels, skipping.", flush=True)
            continue
        vmin, vmax = np.percentile(finite_vals, [2, 98])

        rows = coincident_overpasses(t_f, over_t, args.window_hours)
        drew_any = False
        for ov in rows:
            coincident = ov["min_dt_min"] <= args.window_hours * 60.0
            if not coincident and not args.all_overpasses:
                continue
            obs_slice = (lon_s, lat_s, val_all[ov["k"]])
            if not np.isfinite(obs_slice[2]).any():
                continue
            outpath = os.path.join(
                args.outdir,
                f"asiaaq_{date}_tropomi_{species}_dc8_overlay_ov{ov['k']}.png")
            render(date, species, obs_slice, lon_f, lat_f, alt_f, t_f, ov,
                   extent, vmin, vmax, units, outpath, args.field)
            print(f"  {species} ov{ov['k']} "
                  f"({np.datetime_as_string(ov['overpass_time'], unit='m')}Z): "
                  f"dt={ov['min_dt_min']:.0f}min n_in={ov['n_in_window']} "
                  f"-> {os.path.basename(outpath)}", flush=True)
            n_made += 1
            drew_any = True
        if not drew_any:
            best = rows[0]
            print(f"  {species}: no overpass within ±{args.window_hours}h "
                  f"(closest {best['min_dt_min']:.0f} min). "
                  f"Use --all-overpasses to force.", flush=True)
    return n_made


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    here = os.path.dirname(os.path.abspath(__file__))
    ap.add_argument("--pairdir",
                    default=os.path.join(here, "output", "pair"),
                    help="directory of paired asiaaq_*.nc4 files "
                         "(default: <script>/output/pair)")
    ap.add_argument("--outdir", default=os.path.join(here, "dc8_tropomi_overlays"),
                    help="where to write PNGs")
    ap.add_argument("--window-hours", type=float, default=2.0,
                    help="temporal coincidence half-window, hours (default 2)")
    ap.add_argument("--field", choices=["obs", "model"], default="obs",
                    help="background field: TROPOMI obs column or model column")
    ap.add_argument("--all-overpasses", action="store_true",
                    help="plot every overpass, even without DC8 coincidence")
    ap.add_argument("--dates", nargs="*", default=None,
                    help="restrict to these YYYYMMDD dates")
    args = ap.parse_args(argv)

    if not os.path.isdir(args.pairdir):
        sys.exit(f"pairdir not found: {args.pairdir}")
    os.makedirs(args.outdir, exist_ok=True)

    pairs = find_pairs(args.pairdir)
    if args.dates:
        pairs = {d: v for d, v in pairs.items() if d in set(args.dates)}
    if not pairs:
        sys.exit(f"No dates with both DC8 and TROPOMI pairs in {args.pairdir}")

    print(f"Dates with DC8+TROPOMI coincidence candidates: {', '.join(pairs)}",
          flush=True)
    total = 0
    for date, entry in pairs.items():
        total += process_date(date, entry, args)
    print(f"\nDone. {total} overlay figure(s) written to {args.outdir}", flush=True)


if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined

NameError: name '__file__' is not defined